<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 컴퓨터 비전 · 09. DETR

## End-to-End Object Detection with Transformers

- **원 논문:** [End-to-End Object Detection with Transformers](https://arxiv.org/abs/2005.12872)
- **저자 / 발표:** Nicolas Carion et al. · ECCV 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** object queries, bipartite matching, no-object class, set loss를 재현한다.

**축소하거나 생략한 부분:** ResNet-50, 100 queries, COCO 대신 작은 CNN, 3 queries, 단일 물체를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Eq. (1): Hungarian matching | Task 1 tiny_match | 최소 permutation |
| §3.2 Fig. 2: decoder object queries and image cross-attention | Task 2 explicit query self/cross-attention | [B,H,3,16] cross-attention weights |
| §3.1 Eq. (2) and §3.2: Hungarian set loss/no-object class | Task 3 | set loss; GT-independent objectness query로 held-out class/box 평가 |

## 핵심 아이디어와 수식

$$\hat{\sigma}=\arg\min_{\sigma}\sum_i\mathcal{L}_{match}(y_i,\hat{y}_{\sigma(i)})$$

**기호 해설:** σ는 prediction과 target의 일대일 배정, L_match는 class와 box matching cost다.

**코드 대응:** Task 1이 matching cost, Task 2가 query self-attention과 image cross-attention, Task 3이 set loss를 구현한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → memory [B,16,24] → 3 query class [B,3,5], box [B,3,4].

**포트폴리오 구현 범위:** 단일 물체라 완전 Hungarian 알고리즘은 최소 query 선택으로 축소된다. 학습 matching은 GT를 쓰지만 평가는 no-object 확률만으로 query를 먼저 고른다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Eq. (1): Hungarian matching
- **노트북에서 구현할 부분:** Task 1 tiny_match
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 최소 permutation

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 1: DETR 설정, box L1 cost, 단일-target matching을 구현하세요.
@dataclass
class DETRConfig:
    pass


def box_l1_cost(predicted_boxes, target_boxes):
    raise NotImplementedError


def match_single_target(class_logits, predicted_boxes, labels, target_boxes):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.2 Fig. 2: decoder object queries and image cross-attention
- **노트북에서 구현할 부분:** Task 2 explicit query self/cross-attention
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** [B,H,3,16] cross-attention weights

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 2: CNN memory와 learned object query decoder를 갖는 DETRMini를 구현하세요.
class ExplicitMultiHeadAttention(nn.Module):
    """DETR decoder에서 공유하는 self/cross attention이다."""

    def __init__(self, model_dim, heads):
        super().__init__()

    def split_heads(self, tensor):
        """[B, T, D]를 [B, H, T, d_h]로 분할한다."""
        raise NotImplementedError

    def merge_heads(self, tensor):
        """[B, H, T, d_h]를 [B, T, D]로 병합한다."""
        raise NotImplementedError

    def forward(self, query, key, value, attention_mask=None):
        """Self/cross attention output과 head별 weights를 반환한다."""
        raise NotImplementedError


class DETRDecoderBlock(nn.Module):
    """Object query self-attention과 image cross-attention을 연결한다."""

    def __init__(self, config):
        super().__init__()

    def forward(self, queries, memory):
        """Decoded queries와 image cross-attention weights를 반환한다."""
        raise NotImplementedError


class DETRMini(nn.Module):
    def __init__(self, config):
        super().__init__()

    def encode(self, inputs):
        """CNN feature map을 memory [B, 16, D]로 펼친다."""
        raise NotImplementedError

    def forward(self, inputs):
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Eq. (2) and §3.2: Hungarian set loss/no-object class
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** set loss; GT-independent objectness query로 held-out class/box 평가

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 3: set loss와 GT-independent objectness-selected 평가를 구현하세요.
def detr_set_loss(prediction, labels, target_boxes, config):
    raise NotImplementedError


def train_detr_step(model, optimizer, inputs, labels, boxes, config):
    raise NotImplementedError


def evaluate_detr(model, inputs, labels, boxes):
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.